# Multimodal Financial Market Analysis — BBCA (v3)
**Authors**: Wesley Coa, Geoffrey Gohtama, Wilbert — Bina Nusantara University, 2026

**Install:**
```
pip install neuralprophet pdfplumber scikit-learn matplotlib lightgbm shap optuna
```
**Files needed (same folder as notebook):**
- `BBCA_2015_2025_Combined.csv`
- `monthly_reports/BBCA_YYYY_MM.pdf`

In [29]:
# ============================================================
#  CELL 1 — IMPORTS
# ============================================================
import os, re, glob, warnings, itertools
warnings.filterwarnings('ignore')

import numpy  as np
# NeuralProphet internally calls .view() which only exists on torch tensors,
# not on pandas Series. This patch forces all dataframe columns to plain numpy
# arrays before NeuralProphet touches them.
import pandas as pd
# ─────────────────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

try:
        from neuralprophet import NeuralProphet, set_plotting_backend
        set_plotting_backend('matplotlib')
except ImportError:
    pass

try:
    import pdfplumber
except ImportError:
    raise ImportError('pip install pdfplumber')

try:
    import lightgbm as lgb
except ImportError:
    raise ImportError('pip install lightgbm')


try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError('Run: pip install xgboost')

try:
    import shap
except ImportError:
    raise ImportError('pip install shap')

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    raise ImportError('pip install optuna')

print('All imports OK')

All imports OK


In [30]:
# ============================================================
#  CELL 2 — CONFIGURATION
# ============================================================
PRICE_PATH   = 'BBCA_2015_2025_Combined.csv'
PDF_DIR      = 'monthly_reports'
PARSED_CACHE = 'BBCA_monthly_parsed_v3.csv'

TEST_SIZE    = 240      # ~1 trading year held-out
PUB_LAG_DAYS = 45       # BCA publishes ~4-6 weeks after period end

# XGBoost / LightGBM config
N_LAGS       = 14    # price lag window
XGB_TRIALS   = 30    # Optuna trials

OPTUNA_TRIALS = 30   # reduce to 20 to speed up testing

# Stable ratio features only (raw levels & MoM removed — YTD sawtooth issue)
FUNDAMENTAL_COLS = [
    'net_profit_margin',
    'loans_to_assets',
    'ni_growth_yoy',
    'rev_growth_yoy',
]
# nim_proxy / reported ratios appended automatically if parseable

print('Config OK')

Config OK


In [31]:
# ============================================================
#  CELL 3 — PRICE DATA LOADER
# ============================================================
def load_price_data(path):
    df = pd.read_csv(path)
    for fmt in ('%m/%d/%Y', '%Y-%m-%d', '%d/%m/%Y'):
        try:
            df['Date'] = pd.to_datetime(df['Date'], format=fmt)
            break
        except (ValueError, TypeError):
            continue
    else:
        df['Date'] = pd.to_datetime(df['Date'], infer_datetime_format=True)

    df = (df.sort_values('Date')
            .reset_index(drop=True)
            .rename(columns={'Date': 'ds', 'Close': 'y'})[['ds', 'y']])
    print(f'[Price]  {df["ds"].min().date()} -> {df["ds"].max().date()}'
          f'  |  {len(df):,} trading days')
    return df


In [32]:
# ============================================================
#  CELL 4 — YTD CUMULATIVE -> STANDALONE MONTHLY
#
#  OJK monthly reports are titled "For Periods Ended YYYY-MM-DD"
#  meaning income-statement figures are CUMULATIVE year-to-date.
#  e.g. April 2015 report = Jan+Feb+Mar+Apr combined, not April alone.
#  Balance-sheet items (total_assets, total_loans) are snapshots — no fix.
# ============================================================
_YTD_COLS = ['net_profit', 'interest_income', 'interest_expense', 'personnel_expense']


def ytd_to_monthly(series, dates):
    result = series.copy().astype(float)
    for yr in dates.dt.year.unique():
        mask   = dates.dt.year == yr
        idx    = series.index[mask]
        vals   = series.loc[mask].values.astype(float)
        months = dates.loc[mask].dt.month.values
        order  = np.argsort(months)
        idx, vals, months = idx[order], vals[order], months[order]
        standalone = np.empty_like(vals)
        standalone[0] = vals[0]   # January = standalone (1-month YTD)
        for i in range(1, len(vals)):
            diff = vals[i] - vals[i - 1]
            # If negative (data anomaly), use average-month estimate
            standalone[i] = diff if diff >= 0 else vals[i] / months[i]
        result.loc[idx] = standalone
    return result


In [33]:
# ============================================================
#  CELL 5 — PDF REGEX PATTERNS
#
#  Two OJK layouts exist:
#
#  A) MONTHLY (Jan-Nov): single column, one date.
#     "1. Interest income 14,308,684"
#     "NET PROFIT (LOSS) 5,230,451"
#
#  B) ANNUAL DECEMBER: 4-column layout.
#     "1. Interest income  62,022,745  60,508,105  65,875,355  64,351,925"
#     We extract the FIRST number = Individual (Bank-only) current year.
#     Also has a Financial Ratios page with pre-computed NIM, LDR, ROA, CAR.
# ============================================================

# Monthly + annual income/balance sheet patterns
_PAT_MONTHLY = [
    ('interest_income', [
        r'1\.\s*Interest income\s+([\d,]+)',
        r'1\.\s*Pendapatan Bunga\s+([\d,]+)',
        r'Interest income[^\n]{0,40}([\d,\.]{6,})',
    ]),
    ('interest_expense', [
        r'2\.\s*Interest expenses?\s+([\d,]+)',
        r'2\.\s*Beban Bunga\s+([\d,]+)',
    ]),
    ('net_profit', [
        r'NET PROFIT \(LOSS\) AFTER TAX\s+([\d,]+)',
        r'NET PROFIT \(LOSS\)\s+([\d,]+)',
        r'LABA \(RUGI\) BERSIH\s+([\d,]+)',
        r'NET PROFIT\s+([\d,]{6,})',
    ]),
    ('personnel_expense', [
        r'(?:j\.\s*)?Personnel expenses\s+([\d,]+)',
        r'Beban Tenaga Kerja\s+([\d,]+)',
    ]),
    ('total_assets', [
        r'TOTAL ASSETS\s+([\d,]+)',
        r'TOTAL ASET\s+([\d,]+)',
    ]),
    ('total_loans', [
        r'9\.\s*Loans and financing\s+([\d,]+)',
        r'9\.\s*Loans\s+([\d,]+)',
        r'9\.\s*Kredit\s+([\d,]+)',
    ]),
    ('current_accounts', [
        r'1\.\s*Current account\s+([\d,]+)',
        r'1\.\s*Giro\s+([\d,]+)',
    ]),
    ('savings_accounts', [
        r'2\.\s*Saving account\s+([\d,]+)',
        r'2\.\s*Tabungan\s+([\d,]+)',
    ]),
    ('time_deposits', [
        r'3\.\s*Time deposit\s+([\d,]+)',
        r'3\.\s*Deposito\s+([\d,]+)',
    ]),
]

# December annual report — pre-computed financial ratios page
_PAT_RATIOS = [
    ('nim_reported',  [r'Net Interest Margin\s*\(NIM\)\s+([\d\.]+)%']),
    ('ldr_reported',  [r'Loan to Deposit Ratio\s*\(LDR\)\s+([\d\.]+)%']),
    ('roa_reported',  [r'Return on Asset\s*\(ROA\)\s+([\d\.]+)%']),
    ('roe_reported',  [r'Return on Equity\s*\(ROE\)\s+([\d\.]+)%']),
    ('car_reported',  [
        r'CAR Ratio\s*\(%\)\s+([\d\.]+)%',
        r'Capital Adequacy Ratio\s*\(CAR\)\s+([\d\.]+)%',
    ]),
    ('npl_gross',     [r'Gross NPL\s+([\d\.]+)%']),
    ('bopo_reported', [r'Operating Expenses to Operating Income\s*\(BOPO\)\s+([\d\.]+)%']),
]


def _is_annual_format(text):
    """December annual reports have INDIVIDUAL and CONSOLIDATED columns side-by-side."""
    return bool(
        re.search(r'INDIVIDUAL\s+CONSOLIDATED', text, re.IGNORECASE) or
        re.search(r'Dec 31,\s*\d{4}\s+Dec 31,\s*\d{4}', text)
    )


def _clean_number(raw):
    raw = raw.strip().replace(' ', '').replace(',', '')
    try:
        return float(raw)
    except ValueError:
        return np.nan


def _extract_fields(text, patterns):
    result = {}
    for col, pats in patterns:
        found = np.nan
        for pat in pats:
            m = re.search(pat, text, re.IGNORECASE)
            if m:
                found = _clean_number(m.group(1))
                if not np.isnan(found):
                    break
        result[col] = found
    return result


def _parse_single_pdf(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            full_text = '\n'.join(p.extract_text() or '' for p in pdf.pages)
    except Exception as e:
        print(f'  [WARN] Cannot open {os.path.basename(pdf_path)}: {e}')
        return None

    annual = _is_annual_format(full_text)
    values = _extract_fields(full_text, _PAT_MONTHLY)

    if annual:
        ratios = _extract_fields(full_text, _PAT_RATIOS)
        values.update(ratios)
        values['is_annual'] = 1.0
    else:
        values['is_annual'] = 0.0

    missing = [k for k in ('interest_income', 'net_profit')
               if np.isnan(values.get(k, np.nan))]
    if missing:
        print(f'  [WARN] {missing} not found in {os.path.basename(pdf_path)}')
    return values

print('PDF patterns OK')


PDF patterns OK


In [34]:
# ============================================================
#  CELL 6 — LOAD & CLEAN MONTHLY PDF DATA
# ============================================================
def load_monthly_pdf_data(pdf_dir, cache_path):

    # 1. Load cache
    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path, parse_dates=['ds'])
        cached_months = set(cached['ds'].dt.to_period('M').astype(str))
        print(f'[Monthly]  Cache: {len(cached)} months loaded.')
    else:
        cached = pd.DataFrame()
        cached_months = set()

    # 2. Parse new PDFs
    pdf_files = sorted(glob.glob(os.path.join(pdf_dir, 'BBCA_????_??.pdf')))
    if not pdf_files:
        raise FileNotFoundError(
            f'No PDFs found in "{pdf_dir}". Expected: BBCA_YYYY_MM.pdf')

    new_rows = []
    for pdf_path in pdf_files:
        fname = os.path.basename(pdf_path)
        m = re.match(r'BBCA_(\d{4})_(\d{2})\.pdf', fname)
        if not m:
            continue
        year, month = int(m.group(1)), int(m.group(2))
        period_end  = pd.Timestamp(year, month, 1) + pd.offsets.MonthEnd(0)
        period_key  = period_end.to_period('M').strftime('%Y-%m')
        if period_key in cached_months:
            continue

        print(f'  [Parse] {fname} ...', end=' ')
        values = _parse_single_pdf(pdf_path)
        if values is None:
            continue

        row = {'ds': period_end}
        for col, val in values.items():
            if col == 'is_annual':
                row[col] = val
            else:
                row[col] = val / 1_000 if not np.isnan(val) else np.nan  # M -> B IDR
        new_rows.append(row)
        ni  = row.get('net_profit',      float('nan'))
        rev = row.get('interest_income', float('nan'))
        fmt = 'ANNUAL' if row.get('is_annual') else 'monthly'
        print(f'[{fmt}]  NI={ni:.0f}B  Rev={rev:.0f}B')

    # 3. Merge & save
    if new_rows:
        new_df  = pd.DataFrame(new_rows)
        all_raw = (pd.concat([cached, new_df], ignore_index=True)
                   if not cached.empty else new_df)
        all_raw = all_raw.sort_values('ds').reset_index(drop=True)
        all_raw.to_csv(cache_path, index=False)
        print(f'[Monthly]  {len(new_rows)} new months saved.')
    else:
        all_raw = cached.copy()

    raw = all_raw.sort_values('ds').reset_index(drop=True)

    # 4. YTD -> standalone monthly for income-statement columns
    print('[Monthly]  Converting YTD cumulative -> standalone monthly ...')
    for col in _YTD_COLS:
        if col in raw.columns:
            raw[col] = ytd_to_monthly(raw[col], raw['ds'])

    # 5. Rename
    raw = raw.rename(columns={
        'net_profit':      'net_income',
        'interest_income': 'revenue',
    })

    # 6. Derived features
    raw['net_profit_margin'] = (raw['net_income'] / raw['revenue']).clip(0, 1)

    if 'total_loans' in raw.columns and 'total_assets' in raw.columns:
        lta = raw['total_loans'] / raw['total_assets']
        # Replace near-zero values caused by PDF parse failures
        bad = lta < 0.05
        if bad.any():
            lta[bad] = lta.rolling(12, min_periods=3, center=True).median()[bad]
        raw['loans_to_assets'] = lta
    else:
        raw['loans_to_assets'] = np.nan

    if 'interest_expense' in raw.columns and 'total_assets' in raw.columns:
        raw['nim_proxy'] = (
            (raw['revenue'] - raw['interest_expense']) / raw['total_assets']
        ).clip(0, 0.2)

    if 'nim_reported' in raw.columns:
        raw['nim_reported'] = raw['nim_reported'] / 100  # % -> ratio

    # YoY growth on standalone figures; symmetric clip; MoM removed
    raw['ni_growth_yoy']  = raw['net_income'].pct_change(12).fillna(0).clip(-1, 1)
    raw['rev_growth_yoy'] = raw['revenue'].pct_change(12).fillna(0).clip(-1, 1)

    # 7. Select final columns
    keep = ['ds'] + FUNDAMENTAL_COLS
    for extra in ['nim_proxy', 'nim_reported', 'ldr_reported', 'roa_reported',
                  'car_reported', 'npl_gross', 'bopo_reported']:
        if extra in raw.columns and raw[extra].notna().sum() > 5:
            keep.append(extra)

    fund = raw[[c for c in keep if c in raw.columns]]
    fund = fund.dropna(subset=['net_profit_margin']).reset_index(drop=True)
    feat_cols = [c for c in fund.columns if c != 'ds']
    print(f'[Monthly]  {len(fund)} months ready  '
          f'({fund["ds"].min().date()} -> {fund["ds"].max().date()})')
    print(f'[Monthly]  Features: {feat_cols}')
    return fund


In [35]:
# ============================================================
#  CELL 7 — ALIGN TO DAILY + TRAIN-ONLY NORMALISATION
# ============================================================
def align_to_daily(df_price, fund_m, feat_cols, pub_lag=PUB_LAG_DAYS):
    """
    Forward-fill monthly fundamentals onto trading days.
    Publication lag: a period_end=March 31 report is only available
    after March 31 + 45 days = ~May 15. Prevents look-ahead bias.
    """
    lagged = fund_m.copy()
    lagged['ds'] = lagged['ds'] + pd.DateOffset(days=pub_lag)
    print(f'[Align]  Publication lag = {pub_lag} days applied.')

    calendar = pd.DataFrame({
        'ds': pd.date_range(df_price['ds'].min(), df_price['ds'].max(), freq='D')
    })
    daily = calendar.merge(lagged[['ds'] + feat_cols], on='ds', how='left')
    daily[feat_cols] = daily[feat_cols].ffill()

    first = daily[feat_cols[0]].first_valid_index()
    daily = daily.iloc[first:]

    merged = df_price.merge(daily[['ds'] + feat_cols], on='ds', how='inner')
    merged = merged.dropna(subset=feat_cols).reset_index(drop=True)
    print(f'[Align]  {len(merged):,} trading days  '
          f'({merged["ds"].min().date()} -> {merged["ds"].max().date()})')
    return merged


def fit_normaliser(train_df, cols):
    """Compute Z-score stats from TRAINING rows only — no look-ahead."""
    stats = {}
    for col in cols:
        if col in train_df.columns:
            mu    = train_df[col].mean()
            sigma = train_df[col].std()
            stats[col] = (mu, sigma + 1e-9)
    return stats


def apply_normaliser(df, stats):
    df = df.copy()
    for col, (mu, sigma) in stats.items():
        if col in df.columns:
            df[col] = (df[col] - mu) / sigma
    return df


## Cell 8 — XGBoost Forecaster (replaces NeuralProphet)

In [36]:
"""
================================================================================
  CELL 8 (REPLACEMENT) — XGBoost Forecaster
  Replaces NeuralProphet entirely. No version conflicts.
================================================================================
  Model: XGBoost Regressor using manual lag features (sliding window)
  Two models compared:
    XGB-Baseline  — price lags only (equivalent to NP Baseline)
    XGB-Hybrid    — price lags + monthly fundamental features

  Why XGBoost instead of NeuralProphet:
    • No pandas/torch version conflicts
    • Faster training (seconds not minutes)
    • Naturally handles tabular lag features
    • Same SHAP support as LightGBM
    • Proven strong performance on financial time series
================================================================================
"""

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna

# ── Configuration ─────────────────────────────────────────────────────────────
N_LAGS         = 14     # how many past days the model looks back
XGB_TRIALS     = 30     # Optuna trials for XGBoost HPT


# ==============================================================================
#  HELPER: BUILD LAG FEATURE MATRIX
# ==============================================================================

def make_lag_features(df, n_lags, fund_cols=None):
    """
    Convert a [ds, y, ...fund_cols] DataFrame into a supervised learning matrix.

    For each row t, features are:
      - y_{t-1}, y_{t-2}, ..., y_{t-n_lags}   (price lags)
      - day_of_week, month                      (calendar)
      - fund_cols values at time t              (fundamentals, already forward-filled)

    Target: y_t (next day's close price)

    Returns X (features), y (targets), dates, feature names.
    """
    d = df.copy().sort_values('ds').reset_index(drop=True)
    feat_names = []

    # Price lag columns
    for lag in range(1, n_lags + 1):
        col = f'lag_{lag}'
        d[col] = d['y'].shift(lag)
        feat_names.append(col)

    # Calendar features
    d['dow']   = d['ds'].dt.dayofweek
    d['month'] = d['ds'].dt.month
    feat_names += ['dow', 'month']

    # Fundamental features (already on every row from forward-fill)
    if fund_cols:
        for c in fund_cols:
            if c in d.columns:
                feat_names.append(c)

    # Drop rows where any lag is NaN (first n_lags rows)
    d = d.dropna(subset=feat_names).reset_index(drop=True)

    X     = d[feat_names].values.astype(float)
    y_arr = d['y'].values.astype(float)
    dates = d['ds'].values

    return X, y_arr, dates, feat_names


# ==============================================================================
#  OPTUNA HPT FOR XGBoost REGRESSOR
# ==============================================================================

def xgb_optuna_search(X_tr, y_tr, n_trials=XGB_TRIALS):
    """
    Hyperparameter search for XGBoost using time-series cross-validation.
    Optimises MAE on the last 15% of training data as a validation slice.
    """
    n_val = int(len(X_tr) * 0.15)
    X_t, X_v = X_tr[:-n_val], X_tr[-n_val:]
    y_t, y_v = y_tr[:-n_val], y_tr[-n_val:]

    def objective(trial):
        params = dict(
            n_estimators      = trial.suggest_int('n_estimators', 200, 1000),
            learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
            max_depth         = trial.suggest_int('max_depth', 3, 8),
            subsample         = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 2.0, log=True),
            random_state      = 42,
            verbosity         = 0,
            early_stopping_rounds = 30,
        )
        model = XGBRegressor(**params)
        model.fit(X_t, y_t, eval_set=[(X_v, y_v)], verbose=False)
        pred = model.predict(X_v)
        return mean_absolute_error(y_v, pred)

    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    print(f'[XGB-Optuna]  Best val MAE: {study.best_value:,.1f}')
    print(f'[XGB-Optuna]  Best params : {study.best_params}')
    return study.best_params


# ==============================================================================
#  TRAIN XGBoost MODEL
# ==============================================================================

def train_xgb(df_full, cutoff, fund_cols, best_params, n_lags, name):
    """
    Build lag features, split on cutoff date, train XGBoost, return predictions.

    Returns
    -------
    dict with keys: dates_te, y_true, y_pred, feat_names, X_te, model, metrics
    """
    X, y_arr, dates, feat_names = make_lag_features(df_full, n_lags, fund_cols)

    # Split by date
    train_mask = dates < np.datetime64(cutoff)
    test_mask  = dates >= np.datetime64(cutoff)

    X_tr, y_tr = X[train_mask],  y_arr[train_mask]
    X_te, y_te = X[test_mask],   y_arr[test_mask]
    dates_te   = dates[test_mask]

    print(f'\n  [{name}]  train={len(X_tr):,}  test={len(X_te):,}'
          f'  features={len(feat_names)}')

    # Final model with best params
    params = {**best_params,
              'verbosity': 0, 'random_state': 42}
    params.pop('early_stopping_rounds', None)   # not needed at final fit

    model = XGBRegressor(**params)
    model.fit(X_tr, y_tr, verbose=False)

    y_pred = model.predict(X_te)

    mae  = mean_absolute_error(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mp   = y_te.mean()
    da   = np.mean(np.sign(np.diff(y_te)) == np.sign(np.diff(y_pred))) * 100

    metrics = dict(MAE=mae, RMSE=rmse,
                   MAE_pct=mae/mp*100, RMSE_pct=rmse/mp*100, Dir_Acc=da)

    print(f'     MAE      : {mae:>10,.2f}  ({mae/mp*100:.2f}%)')
    print(f'     RMSE     : {rmse:>10,.2f}  ({rmse/mp*100:.2f}%)')
    print(f'     Dir.Acc  : {da:.2f}%')

    return {
        'dates_te':   dates_te,
        'y_true':     y_te,
        'y_pred':     y_pred,
        'feat_names': feat_names,
        'X_te':       X_te,
        'X_tr':       X_tr,
        'model':      model,
        'metrics':    metrics,
    }


# ==============================================================================
#  SHAP ANALYSIS FOR XGBoost
# ==============================================================================

def run_xgb_shap(res_bl, res_hyb, fund_cols):
    """
    Compute SHAP values for both XGBoost models and produce 3 plots:
      xgb_shap_baseline.png   — beeswarm for price-only model
      xgb_shap_hybrid.png     — beeswarm for hybrid model
      xgb_shap_bar.png        — mean|SHAP| comparison bar chart
    """
    print('\n[SHAP]  Computing XGBoost SHAP values ...')
    exp_bl  = shap.TreeExplainer(res_bl['model'])
    exp_hyb = shap.TreeExplainer(res_hyb['model'])
    sv_bl   = exp_bl.shap_values(res_bl['X_te'])
    sv_hyb  = exp_hyb.shap_values(res_hyb['X_te'])

    feat_bl  = res_bl['feat_names']
    feat_hyb = res_hyb['feat_names']

    # 1. Beeswarm — Baseline
    plt.figure(figsize=(10, 5))
    shap.summary_plot(sv_bl, res_bl['X_te'],
                      feature_names=feat_bl,
                      show=False, plot_type='dot', max_display=15)
    plt.title('SHAP Summary — XGB Baseline (price lags only)', fontsize=11)
    plt.tight_layout()
    plt.savefig('xgb_shap_baseline.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] xgb_shap_baseline.png')

    # 2. Beeswarm — Hybrid
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv_hyb, res_hyb['X_te'],
                      feature_names=feat_hyb,
                      show=False, plot_type='dot', max_display=20)
    plt.title('SHAP Summary — XGB Hybrid (price lags + fundamentals)', fontsize=11)
    plt.tight_layout()
    plt.savefig('xgb_shap_hybrid.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] xgb_shap_hybrid.png')

    # 3. Side-by-side mean |SHAP|
    mean_bl  = np.abs(sv_bl).mean(axis=0)
    mean_hyb = np.abs(sv_hyb).mean(axis=0)
    colors_hyb = ['darkorange' if f in fund_cols else 'steelblue'
                  for f in feat_hyb]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    top = 15
    i_bl  = np.argsort(mean_bl)[-top:]
    i_hyb = np.argsort(mean_hyb)[-top:]

    ax1.barh([feat_bl[i]  for i in i_bl],
             [mean_bl[i]  for i in i_bl],
             color='steelblue', alpha=0.85)
    ax1.set_title('Mean |SHAP| — XGB Baseline')
    ax1.set_xlabel('Mean |SHAP|'); ax1.grid(True, alpha=0.3, axis='x')

    ax2.barh([feat_hyb[i] for i in i_hyb],
             [mean_hyb[i] for i in i_hyb],
             color=[colors_hyb[i] for i in i_hyb], alpha=0.85)
    ax2.set_title('Mean |SHAP| — XGB Hybrid  (orange = fundamentals)')
    ax2.set_xlabel('Mean |SHAP|'); ax2.grid(True, alpha=0.3, axis='x')

    plt.suptitle('XGBoost Feature Importance via SHAP\n'
                 '(orange bars = monthly financial report features)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('xgb_shap_bar.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] xgb_shap_bar.png')


# ==============================================================================
#  COMPARISON PLOT (XGBoost)
# ==============================================================================

def plot_xgb_comparison(df_price, res_bl, res_hyb, test_size):
    fig = plt.figure(figsize=(18, 14))
    fig.suptitle('BBCA — XGBoost Baseline vs Hybrid Multimodal\n'
                 '(Price Lags Only  vs  Price Lags + Monthly Fundamentals)',
                 fontsize=13, fontweight='bold', y=0.99)
    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.50, wspace=0.32)

    cutoff    = df_price['ds'].iloc[-test_size]
    train_act = df_price[df_price['ds'] <  cutoff]
    test_act  = df_price[df_price['ds'] >= cutoff]

    # Panel A — full price history
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(train_act['ds'], train_act['y'],
             color='lightsteelblue', lw=0.8, label='Training Period')
    ax1.plot(test_act['ds'],  test_act['y'],
             'g-', lw=1.2, label='Actual (Test)')
    ax1.plot(pd.to_datetime(res_bl['dates_te']),  res_bl['y_pred'],
             'b--', lw=1.1, label='XGB-Baseline')
    ax1.plot(pd.to_datetime(res_hyb['dates_te']), res_hyb['y_pred'],
             color='darkorange', linestyle='--', lw=1.1, label='XGB-Hybrid')
    ax1.set_title('Full History + Test Period')
    ax1.set_xlabel('Date'); ax1.set_ylabel('Close (IDR)')
    ax1.legend(loc='upper left'); ax1.grid(True, alpha=0.3)

    # Panel B — zoomed test
    ax2 = fig.add_subplot(gs[1, :])
    m_bl  = res_bl['metrics'];  m_hyb = res_hyb['metrics']
    ax2.plot(pd.to_datetime(res_bl['dates_te']), res_bl['y_true'],
             'g.-', lw=1.2, label='Actual')
    ax2.plot(pd.to_datetime(res_bl['dates_te']),  res_bl['y_pred'],
             'b--', lw=1.1,
             label=f"XGB-Baseline  MAE={m_bl['MAE']:,.0f} ({m_bl['MAE_pct']:.1f}%)")
    ax2.plot(pd.to_datetime(res_hyb['dates_te']), res_hyb['y_pred'],
             color='darkorange', linestyle='--', lw=1.1,
             label=f"XGB-Hybrid    MAE={m_hyb['MAE']:,.0f} ({m_hyb['MAE_pct']:.1f}%)")
    ax2.set_title(f'Zoomed — Last {test_size} Trading Days (Test Set)')
    ax2.set_xlabel('Date'); ax2.set_ylabel('Close (IDR)')
    ax2.legend(loc='upper left'); ax2.grid(True, alpha=0.3)

    # Panel C — MAE / RMSE bars
    ax3   = fig.add_subplot(gs[2, 0])
    names = ['XGB-Baseline', 'XGB-Hybrid']
    x, w  = np.arange(2), 0.35
    mae_v  = [m_bl['MAE_pct'],  m_hyb['MAE_pct']]
    rmse_v = [m_bl['RMSE_pct'], m_hyb['RMSE_pct']]
    b1 = ax3.bar(x - w/2, mae_v,  w,
                 color=['steelblue','darkorange'], alpha=0.90, label='MAE (%)')
    b2 = ax3.bar(x + w/2, rmse_v, w,
                 color=['steelblue','darkorange'], alpha=0.55, label='RMSE (%)')
    ax3.bar_label(b1, fmt='%.2f%%', padding=2, fontsize=9)
    ax3.bar_label(b2, fmt='%.2f%%', padding=2, fontsize=9)
    ax3.set_title('Regression Error (lower is better)')
    ax3.set_xticks(x); ax3.set_xticklabels(names)
    ax3.set_ylabel('% of mean price'); ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')

    # Panel D — Directional accuracy
    ax4  = fig.add_subplot(gs[2, 1])
    da_v = [m_bl['Dir_Acc'], m_hyb['Dir_Acc']]
    bars = ax4.bar(names, da_v,
                   color=['steelblue','darkorange'], alpha=0.88, width=0.4)
    ax4.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
    ax4.axhline(50, linestyle='--', color='grey', lw=0.8, label='Random (50%)')
    ax4.set_title('Directional Accuracy (higher is better)')
    ax4.set_ylabel('Accuracy (%)'); ax4.set_ylim(0, 100)
    ax4.legend(); ax4.grid(True, alpha=0.3, axis='y')

    plt.savefig('xgb_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] xgb_comparison.png')


# ==============================================================================
#  SUMMARY TABLE
# ==============================================================================

def print_xgb_summary(res_bl, res_hyb):
    m_bl  = res_bl['metrics']
    m_hyb = res_hyb['metrics']
    dm    = (m_bl['MAE']  - m_hyb['MAE'])  / m_bl['MAE']  * 100
    dr    = (m_bl['RMSE'] - m_hyb['RMSE']) / m_bl['RMSE'] * 100
    dda   = m_hyb['Dir_Acc'] - m_bl['Dir_Acc']

    print('\n' + '='*62)
    print('  XGBoost FINAL RESULTS')
    print('='*62)
    print(f"  {'Metric':<26} {'XGB-Baseline':>16} {'XGB-Hybrid':>16}")
    print(f"  {'-'*58}")
    for label, k, fmt in [
        ('MAE (IDR)',     'MAE',      '{:>15,.2f}'),
        ('MAE (%)',       'MAE_pct',  '{:>14.2f}%'),
        ('RMSE (IDR)',    'RMSE',     '{:>15,.2f}'),
        ('RMSE (%)',      'RMSE_pct', '{:>14.2f}%'),
        ('Dir. Accuracy', 'Dir_Acc',  '{:>14.2f}%'),
    ]:
        print(f"  {label:<26}"
              + fmt.format(m_bl[k])
              + fmt.format(m_hyb[k]))
    print(f"  {'-'*58}")
    print(f"  XGB-Hybrid vs Baseline  ->  "
          f"MAE {dm:+.2f}%   RMSE {dr:+.2f}%   Dir.Acc {dda:+.2f}pp")
    print('='*62)


print('XGBoost functions defined OK')

XGBoost functions defined OK


## Cells 9–11 — Price Feature Engineering + LightGBM + SHAP

In [37]:
# ============================================================
#  CELL 9 — PRICE FEATURE ENGINEERING (for LightGBM)
# ============================================================
PRICE_FEATS = [
    'ret_1d', 'ret_5d', 'ret_10d', 'ret_20d',
    'px_vs_ma5', 'px_vs_ma20',
    'vol_10d', 'vol_20d', 'rsi14',
    'dow', 'month',
]


def engineer_price_features(df):
    d = df.copy().sort_values('ds').reset_index(drop=True)
    y = d['y']

    d['ret_1d']  = y.pct_change(1)
    d['ret_5d']  = y.pct_change(5)
    d['ret_10d'] = y.pct_change(10)
    d['ret_20d'] = y.pct_change(20)

    ma5  = y.rolling(5).mean()
    ma20 = y.rolling(20).mean()
    d['px_vs_ma5']  = y / ma5  - 1
    d['px_vs_ma20'] = y / ma20 - 1

    d['vol_10d'] = d['ret_1d'].rolling(10).std()
    d['vol_20d'] = d['ret_1d'].rolling(20).std()

    delta = y.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    d['rsi14'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    d['dow']   = d['ds'].dt.dayofweek
    d['month'] = d['ds'].dt.month

    d['target'] = (y.shift(-1) > y).astype(int)  # 1=up, 0=down
    return d


In [38]:
# ============================================================
#  CELL 10 — OPTUNA HPT FOR LIGHTGBM + TRAINING
#
#  TimeSeriesSplit: NO data shuffling (shuffled CV on time series
#  is look-ahead bias). Optimises directional accuracy on CV folds.
# ============================================================
def lgb_optuna_search(X, y, n_trials=OPTUNA_TRIALS, n_splits=5):
    tscv = TimeSeriesSplit(n_splits=n_splits)

    def objective(trial):
        params = dict(
            objective         = 'binary',
            metric            = 'binary_logloss',
            verbose           = -1,
            n_estimators      = trial.suggest_int('n_estimators', 100, 800),
            learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            num_leaves        = trial.suggest_int('num_leaves', 15, 63),
            min_child_samples = trial.suggest_int('min_child_samples', 20, 80),
            subsample         = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 1.0, log=True),
            reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 2.0, log=True),
            random_state      = 42,
        )
        accs = []
        for tr_idx, va_idx in tscv.split(X):
            X_tr, X_va = X[tr_idx], X[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]
            clf = lgb.LGBMClassifier(**params)
            clf.fit(X_tr, y_tr,
                    eval_set=[(X_va, y_va)],
                    callbacks=[lgb.early_stopping(30, verbose=False),
                               lgb.log_evaluation(period=-1)])
            accs.append((clf.predict(X_va) == y_va).mean())
        return np.mean(accs)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'[Optuna]  Best CV Dir.Acc: {study.best_value*100:.2f}%')
    print(f'[Optuna]  Best params: {study.best_params}')
    return study.best_params


def train_lgb(train_df, test_df, fund_cols, best_params=None,
              use_fund=True, name='LGB'):
    feat_cols = PRICE_FEATS + (fund_cols if use_fund else [])
    feat_cols = [c for c in feat_cols if c in train_df.columns]

    X_tr = train_df[feat_cols].fillna(0).values
    y_tr = train_df['target'].values
    X_te = test_df[feat_cols].fillna(0).values
    y_te = test_df['target'].values

    if best_params:
        params = {**best_params,
                  'objective': 'binary', 'verbose': -1, 'random_state': 42}
    else:
        params = dict(objective='binary', n_estimators=500, learning_rate=0.02,
                      num_leaves=31, min_child_samples=30, subsample=0.8,
                      colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.5,
                      random_state=42, verbose=-1)

    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr,
              eval_set=[(X_te, y_te)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])

    pred = model.predict(X_te)
    da   = (pred == y_te).mean() * 100

    fi = pd.DataFrame({'feature': feat_cols,
                       'importance': model.feature_importances_})
    fi = fi.sort_values('importance', ascending=False).reset_index(drop=True)

    print(f'  -- {name}  |  Dir.Acc = {da:.2f}%  (trees: {model.best_iteration_})')
    return da, fi, model, feat_cols, X_te, y_te


In [39]:
# ============================================================
#  CELL 11 — SHAP ANALYSIS
#
#  Answers thesis RQ3: "Which financial report features drive
#  the model's directional predictions, and by how much?"
#
#  Outputs:
#    shap_summary_baseline.png   - beeswarm, price-only model
#    shap_summary_hybrid.png     - beeswarm, hybrid model
#    shap_bar_comparison.png     - mean|SHAP| side-by-side bar
#    shap_dependence_fundamentals.png - how each fundamental
#                                       affects prediction
# ============================================================
def run_shap_analysis(model_bl, model_hyb,
                      X_te_bl,  X_te_hyb,
                      feat_bl,  feat_hyb,
                      fund_cols):
    print('\n[SHAP]  Computing SHAP values ...')
    exp_bl  = shap.TreeExplainer(model_bl)
    exp_hyb = shap.TreeExplainer(model_hyb)
    sv_bl   = exp_bl.shap_values(X_te_bl)
    sv_hyb  = exp_hyb.shap_values(X_te_hyb)

    # Binary classifiers return [class0_shap, class1_shap]; take class1
    if isinstance(sv_bl, list):
        sv_bl  = sv_bl[1]
        sv_hyb = sv_hyb[1]

    # 1. Beeswarm — Baseline
    plt.figure(figsize=(10, 6))
    shap.summary_plot(sv_bl, X_te_bl, feature_names=feat_bl,
                      show=False, plot_type='dot', max_display=15)
    plt.title('SHAP Summary — LGB Baseline (price features only)', fontsize=11)
    plt.tight_layout()
    plt.savefig('shap_summary_baseline.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] shap_summary_baseline.png')

    # 2. Beeswarm — Hybrid
    plt.figure(figsize=(10, 7))
    shap.summary_plot(sv_hyb, X_te_hyb, feature_names=feat_hyb,
                      show=False, plot_type='dot', max_display=20)
    plt.title('SHAP Summary — LGB Hybrid (price + fundamentals)', fontsize=11)
    plt.tight_layout()
    plt.savefig('shap_summary_hybrid.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] shap_summary_hybrid.png')

    # 3. Bar chart — mean |SHAP| comparison
    mean_bl  = np.abs(sv_bl).mean(axis=0)
    mean_hyb = np.abs(sv_hyb).mean(axis=0)
    colors_hyb = ['darkorange' if f in fund_cols else 'steelblue'
                  for f in feat_hyb]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    top_n = 15
    idx_bl  = np.argsort(mean_bl)[-top_n:]
    idx_hyb = np.argsort(mean_hyb)[-top_n:]

    ax1.barh([feat_bl[i]  for i in idx_bl],
             [mean_bl[i]  for i in idx_bl], color='steelblue', alpha=0.85)
    ax1.set_title('Mean |SHAP| — Baseline', fontsize=10)
    ax1.set_xlabel('Mean |SHAP value|'); ax1.grid(True, alpha=0.3, axis='x')

    ax2.barh([feat_hyb[i] for i in idx_hyb],
             [mean_hyb[i] for i in idx_hyb],
             color=[colors_hyb[i] for i in idx_hyb], alpha=0.85)
    ax2.set_title('Mean |SHAP| — Hybrid  (orange = fundamental features)', fontsize=10)
    ax2.set_xlabel('Mean |SHAP value|'); ax2.grid(True, alpha=0.3, axis='x')

    plt.suptitle('Feature Importance via SHAP', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_bar_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] shap_bar_comparison.png')

    # 4. Dependence plots for each fundamental feature
    fund_in_hyb = [f for f in fund_cols if f in feat_hyb]
    if fund_in_hyb:
        n_dep = min(4, len(fund_in_hyb))
        fig, axes = plt.subplots(1, n_dep, figsize=(5 * n_dep, 4))
        if n_dep == 1:
            axes = [axes]
        for ax, feat in zip(axes, fund_in_hyb[:n_dep]):
            fi_idx = feat_hyb.index(feat)
            sc = ax.scatter(X_te_hyb[:, fi_idx], sv_hyb[:, fi_idx],
                            c=sv_hyb[:, fi_idx], cmap='RdBu',
                            alpha=0.5, s=8)
            ax.axhline(0, color='grey', lw=0.5, linestyle='--')
            ax.set_xlabel(feat, fontsize=9)
            ax.set_ylabel('SHAP value', fontsize=9)
            ax.set_title(f'Dependence: {feat}', fontsize=9)
            ax.grid(True, alpha=0.3)
            plt.colorbar(sc, ax=ax)
        plt.tight_layout()
        plt.savefig('shap_dependence_fundamentals.png', dpi=150, bbox_inches='tight')
        plt.close()
        print('[Saved] shap_dependence_fundamentals.png')

    # 5. Print fundamental SHAP rankings
    print('\n[SHAP]  Fundamental feature contributions in Hybrid model:')
    all_ranks = sorted(range(len(mean_hyb)),
                       key=lambda x: mean_hyb[x], reverse=True)
    for f in fund_in_hyb:
        fi_idx   = feat_hyb.index(f)
        rank     = all_ranks.index(fi_idx) + 1
        print(f'  {f:<30}  mean|SHAP|={mean_hyb[fi_idx]:.5f}  '
              f'rank={rank}/{len(feat_hyb)}')


## Cell 14 — Main (runs all models)

In [40]:
"""
================================================================================
  CELL 14 (REPLACEMENT) — MAIN
  Runs LightGBM (directional) + XGBoost (regression) hybrid pipeline.
  NeuralProphet completely removed.
================================================================================
"""

def main():

    # ── 1. Load data ──────────────────────────────────────────────────────────
    df_price = load_price_data(PRICE_PATH)
    fund_m   = load_monthly_pdf_data(PDF_DIR, PARSED_CACHE)
    avail_cols = [c for c in fund_m.columns if c != 'ds']
    print(f'[Config]  Fundamental features: {avail_cols}')

    # ── 2. Align fundamentals → daily (with publication lag) ─────────────────
    df_full = align_to_daily(df_price, fund_m, avail_cols)
    df_bl   = df_full[['ds', 'y']].copy()

    # ── 3. Train-only normalisation (no look-ahead) ───────────────────────────
    cutoff    = df_full['ds'].iloc[-TEST_SIZE]
    train_hyb = df_full[df_full['ds'] < cutoff]
    norm_stats = fit_normaliser(train_hyb, avail_cols)
    df_full_n  = apply_normaliser(df_full, norm_stats)

    print(f'\n[Split]  Cutoff: {cutoff.date()}'
          f'  |  Train: {(df_bl["ds"] < cutoff).sum():,}'
          f'  |  Test: {TEST_SIZE}')

    # ── 4. XGBoost HPT (on hybrid feature space, training data only) ─────────
    print('\n[Step 4]  XGBoost Optuna HPT ...')
    X_opt, y_opt, _, _ = make_lag_features(
        df_full_n[df_full_n['ds'] < cutoff], N_LAGS, avail_cols
    )
    best_xgb_params = xgb_optuna_search(X_opt, y_opt, n_trials=XGB_TRIALS)

    # ── 5. Train XGBoost — Baseline (price lags only) ─────────────────────────
    print('\n[Step 5]  Training XGBoost models ...')
    print('\n  XGB-Baseline:')
    res_xgb_bl = train_xgb(
        df_bl, cutoff,
        fund_cols  = None,
        best_params= best_xgb_params,
        n_lags     = N_LAGS,
        name       = 'XGB-Baseline',
    )

    # ── 6. Train XGBoost — Hybrid (price lags + fundamentals) ─────────────────
    print('\n  XGB-Hybrid:')
    res_xgb_hyb = train_xgb(
        df_full_n, cutoff,
        fund_cols  = avail_cols,
        best_params= best_xgb_params,
        n_lags     = N_LAGS,
        name       = 'XGB-Hybrid',
    )

    # ── 7. LightGBM feature engineering ───────────────────────────────────────
    print('\n[Step 7]  Engineering features for LightGBM ...')
    feat_bl_df  = engineer_price_features(df_bl)
    feat_hyb_df = engineer_price_features(df_full_n)

    feat_bl_df  = feat_bl_df.dropna(
        subset=PRICE_FEATS + ['target']).reset_index(drop=True)
    feat_hyb_df = feat_hyb_df.dropna(
        subset=PRICE_FEATS + ['target']).reset_index(drop=True)

    def split_lgb(df):
        idx = df['ds'].searchsorted(cutoff)
        return df.iloc[:idx].copy(), df.iloc[idx:].copy()

    tr_bl_lgb,  te_bl_lgb  = split_lgb(feat_bl_df)
    tr_hyb_lgb, te_hyb_lgb = split_lgb(feat_hyb_df)

    # ── 8. Optuna HPT for LightGBM ────────────────────────────────────────────
    print('\n[Step 8]  Optuna HPT for LightGBM ...')
    hyb_feat_lgb = PRICE_FEATS + [c for c in avail_cols if c in tr_hyb_lgb.columns]
    X_lgb_opt = tr_hyb_lgb[hyb_feat_lgb].fillna(0).values
    y_lgb_opt = tr_hyb_lgb['target'].values
    best_lgb_params = lgb_optuna_search(X_lgb_opt, y_lgb_opt, n_trials=OPTUNA_TRIALS)

    # ── 9. Train final LightGBM models ────────────────────────────────────────
    print('\n[Step 9]  Training final LightGBM models ...')
    da_lgb_bl, fi_bl, lgb_bl, feat_bl_cols, X_te_bl, y_te_bl = train_lgb(
        tr_bl_lgb,  te_bl_lgb,  [],
        best_params=best_lgb_params, use_fund=False, name='LGB-Baseline')

    da_lgb_hyb, fi_hyb, lgb_hyb, feat_hyb_cols, X_te_hyb, y_te_hyb = train_lgb(
        tr_hyb_lgb, te_hyb_lgb, avail_cols,
        best_params=best_lgb_params, use_fund=True, name='LGB-Hybrid')

    lgb_da = {'LGB-Baseline': da_lgb_bl, 'LGB-Hybrid': da_lgb_hyb}

    # ── 10. SHAP analysis ─────────────────────────────────────────────────────
    print('\n[Step 10]  SHAP analysis — LightGBM ...')
    run_shap_analysis(lgb_bl, lgb_hyb,
                      X_te_bl, X_te_hyb,
                      feat_bl_cols, feat_hyb_cols,
                      avail_cols)

    print('\n[Step 10b]  SHAP analysis — XGBoost ...')
    run_xgb_shap(res_xgb_bl, res_xgb_hyb, avail_cols)

    # ── 11. Plots ─────────────────────────────────────────────────────────────
    print('\n[Step 11]  Generating comparison plots ...')
    plot_xgb_comparison(df_bl, res_xgb_bl, res_xgb_hyb, TEST_SIZE)

    # Combined directional accuracy chart (all 4 models)
    all_da = {
        'XGB-Baseline': res_xgb_bl['metrics']['Dir_Acc'],
        'XGB-Hybrid':   res_xgb_hyb['metrics']['Dir_Acc'],
        'LGB-Baseline': da_lgb_bl,
        'LGB-Hybrid':   da_lgb_hyb,
    }
    fig, ax = plt.subplots(figsize=(10, 5))
    colors  = ['steelblue','darkorange','steelblue','darkorange']
    hatches = ['','','//','//']
    bars = ax.bar(list(all_da.keys()), list(all_da.values()),
                  color=colors, alpha=0.85, width=0.5, edgecolor='white')
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=10)
    ax.axhline(50, linestyle='--', color='grey', lw=0.8, label='Random baseline (50%)')
    ax.set_title('Directional Accuracy — All Models\n'
                 '(solid = XGBoost, hatched = LightGBM | '
                 'blue = Baseline, orange = Hybrid)',
                 fontsize=11)
    ax.set_ylabel('Accuracy (%)'); ax.set_ylim(35, 75)
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('all_models_directional_accuracy.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('[Saved] all_models_directional_accuracy.png')

    # ── 12. Summary ───────────────────────────────────────────────────────────
    print_xgb_summary(res_xgb_bl, res_xgb_hyb)

    print('\n  Directional Accuracy — LightGBM:')
    for name, da in lgb_da.items():
        print(f'    {name:<30}: {da:.2f}%')

    print('\n  Output files:')
    for f in ['xgb_comparison.png', 'xgb_shap_baseline.png',
              'xgb_shap_hybrid.png', 'xgb_shap_bar.png',
              'shap_summary_baseline.png', 'shap_summary_hybrid.png',
              'shap_bar_comparison.png', 'all_models_directional_accuracy.png']:
        exists = '✓' if os.path.exists(f) else '✗'
        print(f'    [{exists}] {f}')


main()

[Price]  2015-01-02 -> 2025-12-30  |  2,659 trading days
[Monthly]  Cache: 106 months loaded.
[Monthly]  Converting YTD cumulative -> standalone monthly ...
[Monthly]  104 months ready  (2015-03-31 -> 2025-11-30)
[Monthly]  Features: ['net_profit_margin', 'loans_to_assets', 'ni_growth_yoy', 'rev_growth_yoy', 'nim_proxy']
[Config]  Fundamental features: ['net_profit_margin', 'loans_to_assets', 'ni_growth_yoy', 'rev_growth_yoy', 'nim_proxy']
[Align]  Publication lag = 45 days applied.
[Align]  2,568 trading days  (2015-05-15 -> 2025-12-30)

[Split]  Cutoff: 2024-12-23  |  Train: 2,328  |  Test: 240

[Step 4]  XGBoost Optuna HPT ...
[XGB-Optuna]  Best val MAE: 680.7
[XGB-Optuna]  Best params : {'n_estimators': 707, 'learning_rate': 0.026185681719742622, 'max_depth': 8, 'subsample': 0.7058669797110755, 'colsample_bytree': 0.7947992509228796, 'reg_alpha': 0.8087411777442793, 'reg_lambda': 0.017159776395078995}

[Step 5]  Training XGBoost models ...

  XGB-Baseline:

  [XGB-Baseline]  train=

Best trial: 4. Best value: 0.545312: 100%|██████████| 30/30 [00:03<00:00,  8.17it/s]


[Optuna]  Best CV Dir.Acc: 54.53%
[Optuna]  Best params: {'n_estimators': 145, 'learning_rate': 0.08580222514877409, 'num_leaves': 62, 'min_child_samples': 69, 'subsample': 0.7218455076693483, 'colsample_bytree': 0.5488360570031919, 'reg_alpha': 0.11290133559092672, 'reg_lambda': 0.02837635341868618}

[Step 9]  Training final LightGBM models ...
  -- LGB-Baseline  |  Dir.Acc = 57.08%  (trees: 13)
  -- LGB-Hybrid  |  Dir.Acc = 57.92%  (trees: 15)

[Step 10]  SHAP analysis — LightGBM ...

[SHAP]  Computing SHAP values ...
[Saved] shap_summary_baseline.png
[Saved] shap_summary_hybrid.png
[Saved] shap_bar_comparison.png
[Saved] shap_dependence_fundamentals.png

[SHAP]  Fundamental feature contributions in Hybrid model:
  net_profit_margin               mean|SHAP|=0.02640  rank=8/16
  loans_to_assets                 mean|SHAP|=0.01134  rank=16/16
  ni_growth_yoy                   mean|SHAP|=0.01451  rank=14/16
  rev_growth_yoy                  mean|SHAP|=0.02488  rank=9/16
  nim_proxy      

## Appendix A — Debug PDF

In [41]:
# Change to any PDF path to debug
DEBUG_PDF = 'monthly_reports/BBCA_2015_08.pdf'

with pdfplumber.open(DEBUG_PDF) as pdf:
    raw_text = '\n'.join(p.extract_text() or '' for p in pdf.pages)

annual = _is_annual_format(raw_text)
print(f'Format: {"ANNUAL (December)" if annual else "MONTHLY"}')
print('\n=== RAW TEXT (first 2000 chars) ===')
print(raw_text[:2000])

print('\n=== EXTRACTED VALUES ===')
vals = _extract_fields(raw_text, _PAT_MONTHLY)
if annual:
    vals.update(_extract_fields(raw_text, _PAT_RATIOS))

for k, v in vals.items():
    if k == 'is_annual':
        continue
    try:
        vf = float(v)
        if not np.isnan(vf):
            print(f'  {k:<28}: {vf:>15,.0f} M IDR  ({vf/1000:>10,.1f} B IDR)')
        else:
            print(f'  {k:<28}: NOT FOUND')
    except Exception:
        print(f'  {k:<28}: {v}')

# Expected for BBCA_2015_08.pdf:
# interest_income : 28,772,308 M IDR
# net_profit      : 11,724,925 M IDR
# total_assets    : 569,985,436 M IDR
# total_loans     : 358,988,534 M IDR
# interest_expense:  7,356,116 M IDR


Format: MONTHLY

=== RAW TEXT (first 2000 chars) ===
PT BANK CENTRAL ASIA Tbk
STATEMENTS OF FINANCIAL POSITION
As of August 31, 2015
(In millions of Rupiah)
BANK
No. ACCOUNTS Unaudited
August 31, 2015
ASSETS
1. Cash 13,815,178
2. Placement to Bank Indonesia 68,082,675
3. Interbank placement 13,076,870
4. Spot and derivatives claims 26,646
5. Securities 67,292,481
a. Measured at fair value through profit and loss 1,137,387
b. Available for sale 49,059,656
c. Held to maturity 14,255,063
d. Loan and receivables 2,840,375
6. Securities sold under repurchase agreement
(repo) -
7. Claims on securities bought under reverse
repo 27,354,383
8. Acceptance claims 6,444,849
9. Loans 358,988,534
a. Measured at fair value through profit and loss -
b. Available for sale -
c. Held to maturity -
d. Loan and receivables 358,988,534
10. Sharia Financing -
11. Equity investment 1,848,475
12. Impairment on financial assets -/- (8,187,231)
a. Securities (737,810)
b. Loans (7,209,370)
c. Others (240,051)
13.

## Appendix B — YTD check

In [42]:
fund_check = load_monthly_pdf_data(PDF_DIR, PARSED_CACHE)
print(fund_check[['ds', 'net_profit_margin', 'ni_growth_yoy', 'loans_to_assets']]
      .head(36).to_string())

plt.figure(figsize=(12, 3))
plt.plot(fund_check['ds'], fund_check['net_profit_margin'],
         marker='o', markersize=3, lw=1, color='steelblue')
plt.title('Net Profit Margin after YTD fix — should be smooth (no sawtooth)')
plt.ylim(0, 1); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ytd_check.png', dpi=120, bbox_inches='tight')
plt.close()
print('[Saved] ytd_check.png')


[Monthly]  Cache: 106 months loaded.
[Monthly]  Converting YTD cumulative -> standalone monthly ...
[Monthly]  104 months ready  (2015-03-31 -> 2025-11-30)
[Monthly]  Features: ['net_profit_margin', 'loans_to_assets', 'ni_growth_yoy', 'rev_growth_yoy', 'nim_proxy']
           ds  net_profit_margin  ni_growth_yoy  loans_to_assets
0  2015-03-31       2.788092e-07       0.000000         0.614962
1  2015-04-30       1.000000e+00       0.000000         0.620557
2  2015-05-31       4.461470e-01       0.000000         0.622330
3  2015-06-30       3.961891e-01       0.000000         0.619548
4  2015-07-31       4.869979e-01       0.000000         0.620524
5  2015-08-31       4.648391e-01       0.000000         0.629821
6  2015-09-30       3.305668e-01       0.000000         0.636675
7  2015-10-31       4.757423e-01       0.000000         0.648404
8  2015-11-30       5.276968e-01       0.000000         0.647370
9  2016-01-31       4.229590e-01       0.000000         0.645475
10 2016-02-29      